## Qwen3 本地部署

本 Notebook 演示如何在本地部署 Qwen3 系列模型 [(Hugging Face)](https://huggingface.co/collections/Qwen/qwen3)，并实现多轮对话（需预先配置 GPU 版 PyTorch 环境）

`pip install transformers>=4.51.0 accelerate>= 1.1.0`

### 1. 加载模型和 Tokenizer

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")

print(f"模型加载完成，设备：{model.device}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

模型加载完成，设备：cuda:0


### 2. 定义对话函数

In [2]:
def chat(messages: list) -> str:
    """
    单次推理，输入完整历史消息列表，返回模型回复字符串。
    messages 格式：[{"role": "user"/"assistant", "content": "..."}]
    """
    # 将消息列表转换为模型输入格式
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # 在对话文本末尾添加一个提示模型开始生成回复的标记
        enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # 生成回复
    output_ids = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        do_sample=True
    )

    # 只解码新生成的 token
    new_ids = output_ids[0][len(inputs.input_ids[0]):]
    return tokenizer.decode(new_ids, skip_special_tokens=True)

### 3. 多轮对话主循环

运行后在输入框中输入问题，输入 `quit` 退出

In [3]:
# 设置系统提示词
SYSTEM_PROMPT = "You are a helpful assistant."  # 改为 "" 即不使用

# 初始化对话历史
history = []
if SYSTEM_PROMPT:
    history.append({"role": "system", "content": SYSTEM_PROMPT})

while True:
    user_input = input("你：").strip()
    if user_input.lower() == "quit":
        print("\n对话结束")
        break
    if not user_input:
        continue

    # 追加用户消息
    history.append({"role": "user", "content": user_input})

    # 获取回复
    reply = chat(history)
    print(f"\nQwen3：{reply}\n")

    # 追加助手消息
    history.append({"role": "assistant", "content": reply})

你： 你是谁？


C:\Users\22418\anaconda3\envs\Scorpio\Lib\site-packages\transformers\integrations\sdpa_attention.py:54: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(



Qwen3：我是你的助手，我叫小助手。我是一个AI助手，可以帮助你回答问题、提供信息、帮助你完成任务等等。我希望能为你提供有用的信息和帮助。如果你有任何问题或需要帮助，随时告诉我。



你： Attention是什么？



Qwen3："Attention" 是一个在人工智能和机器学习领域非常重要的概念，尤其是在 **Transformer** 模型中。它指的是模型在处理输入数据时，对某些部分（如某些词或特征）给予更多的关注或强调。

### 什么是 Attention？
**Attention**（注意力机制）是一种让模型能够“关注”到输入中重要部分的机制。它允许模型在处理信息时，动态地分配注意力，从而更有效地捕捉到关键信息。

### 常见的 Attention 类型
1. **Self-Attention（自注意力）**：
   - 模型在处理每个词时，会考虑与之相关的所有词（包括自身）。
   - 例如，在语言模型中，每个词的表示会与所有其他词的表示进行比较，以确定哪些词更重要。

2. **Cross-Attention（交叉注意力）**：
   - 用于不同输入之间的关系，例如在机器翻译中，源语言和目标语言之间的关系。

3. **Multi-Head Attention（多头注意力）**：
   - 通过多个注意力头来捕捉不同方面的信息，增强模型的表达能力。

### 为什么 Attention 重要？
- **提升模型性能**：Attention 机制可以让模型更有效地捕捉上下文信息，提高模型的准确性和泛化能力。
- **处理长序列**：在处理长文本时，Attention 能够帮助模型动态地关注关键部分，而不是逐个处理。
- **增强模型理解**：Attention 让模型能够理解输入中的关系和依赖，从而更好地生成输出。

### 举例说明
假设你在翻译句子：“The cat sat on the mat.”：
- 模型会通过 Attention 机制关注“cat”、“sat”、“on”、“mat”等词，从而理解句子的整体意思。

### 总结
**Attention** 是一种让模型能够动态关注输入中重要部分的机制，它在现代深度学习模型（如 Transformer）中起着至关重要的作用。通过 Attention，模型可以更有效地捕捉上下文信息，从而提高性能和理解能力。

如果你对 Attention 有更具体的问题，欢迎继续提问！



你： quit



对话结束
